# Data Exploration

EDA and visualizations for the HDB resale dataset. Run `1_misc_features.ipynb` first to produce the input files.

## Inputs
- `../../merged_data/merged_hdb_resale_with_rpi.csv`
- `../../merged_data/quarterly_summary.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

merged_df = pd.read_csv('../../merged_data/merged_hdb_resale_with_rpi.csv', parse_dates=['month'])
quarterly_summary = pd.read_csv('../../merged_data/quarterly_summary.csv')

print(f"Loaded {len(merged_df):,} transactions across {merged_df['quarter'].nunique()} quarters")

Loaded 222,077 transactions across 36 quarters


## Visualizations

In [ ]:
# Distribution of resale prices (nominal vs real)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(merged_df['resale_price'] / 1000, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Nominal Resale Prices', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Resale Price (SGD thousands)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(merged_df['resale_price'].median() / 1000, color='red', linestyle='--', label=f"Median: ${merged_df['resale_price'].median()/1000:.0f}K")
axes[0].legend()

axes[1].hist(merged_df['resale_price_real'] / 1000, bins=60, color='orange', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution of Real Resale Prices (2025-Q4 base)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Real Resale Price (SGD thousands)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(merged_df['resale_price_real'].median() / 1000, color='red', linestyle='--', label=f"Median: ${merged_df['resale_price_real'].median()/1000:.0f}K")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Time series: Median nominal vs real resale prices and RPI
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Nominal vs Real median prices over time
axes[0].plot(quarterly_summary['quarter'], quarterly_summary['median_price'] / 1000,
             marker='o', linewidth=2, color='steelblue', label='Nominal')
axes[0].plot(quarterly_summary['quarter'], quarterly_summary['median_real_price'] / 1000,
             marker='s', linewidth=2, color='orange', label='Real (2025-Q4 base)')
axes[0].set_title('Median HDB Resale Price Over Time (Nominal vs Real)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price (SGD thousands)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: RPI over time
axes[1].plot(quarterly_summary['quarter'], quarterly_summary['rpi'],
             marker='D', linewidth=2, color='darkred')
axes[1].set_title('HDB Resale Price Index (RPI) Over Time', fontsize=12, fontweight='bold')
axes[1].set_ylabel('RPI (2009-Q1 = 100)')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

# Plot 3: Transaction volume over time
axes[2].bar(quarterly_summary['quarter'], quarterly_summary['num_transactions'],
            color='seagreen', edgecolor='white', alpha=0.8)
axes[2].set_title('Number of Resale Transactions Per Quarter', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Transactions')
axes[2].set_xlabel('Quarter')
axes[2].grid(True, alpha=0.3)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Resale price distribution by flat type
fig, ax = plt.subplots(figsize=(12, 6))

flat_type_order = ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']
plot_data = merged_df[merged_df['flat_type'].isin(flat_type_order)]

sns.boxplot(data=plot_data, x='flat_type', y='resale_price',
            order=flat_type_order, palette='viridis', ax=ax,
            flierprops=dict(marker='.', markersize=1, alpha=0.3))
ax.set_title('Resale Price Distribution by Flat Type', fontsize=12, fontweight='bold')
ax.set_xlabel('Flat Type')
ax.set_ylabel('Resale Price (SGD)')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Median resale price by town (horizontal bar chart)
fig, ax = plt.subplots(figsize=(12, 8))

town_median = merged_df.groupby('town')['resale_price'].median().sort_values()
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(town_median)))

town_median.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Median Resale Price by Town', fontsize=12, fontweight='bold')
ax.set_xlabel('Median Resale Price (SGD)')
ax.set_ylabel('')
ax.axvline(merged_df['resale_price'].median(), color='red', linestyle='--', alpha=0.7, label=f"Overall Median: ${merged_df['resale_price'].median():,.0f}")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric variables
fig, ax = plt.subplots(figsize=(10, 8))

numeric_cols = ['floor_area_sqm', 'lease_commence_date', 'resale_price', 'rpi', 'resale_price_real']
corr_matrix = merged_df[numeric_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.3f', square=True, linewidths=1, ax=ax)
ax.set_title('Correlation Matrix of Numeric Variables', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey observations:")
print(f"   - Floor area vs Resale price: {corr_matrix.loc['floor_area_sqm', 'resale_price']:.3f}")
print(f"   - Lease commence date vs Resale price: {corr_matrix.loc['lease_commence_date', 'resale_price']:.3f}")
print(f"   - RPI vs Resale price: {corr_matrix.loc['rpi', 'resale_price']:.3f}")

In [ ]:
# Resale price vs floor area, colored by flat type
fig, ax = plt.subplots(figsize=(12, 7))

flat_types = ['2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE']
palette = sns.color_palette('Set2', len(flat_types))

for ft, color in zip(flat_types, palette):
    subset = merged_df[merged_df['flat_type'] == ft].sample(min(1000, len(merged_df[merged_df['flat_type'] == ft])), random_state=42)
    ax.scatter(subset['floor_area_sqm'], subset['resale_price'] / 1000,
               alpha=0.3, s=10, label=ft, color=color)

ax.set_title('Resale Price vs Floor Area by Flat Type', fontsize=12, fontweight='bold')
ax.set_xlabel('Floor Area (sqm)')
ax.set_ylabel('Resale Price (SGD thousands)')
ax.legend(title='Flat Type', markerscale=3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()